### Install required libraries

In [24]:
!pip3 install openai
!pip3 install requests
!pip3 install python-dotenv

  Using cached requests-2.32.4-py3-none-any.whl (64 kB)
  Using cached charset_normalizer-3.4.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (149 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl (129 kB)


### Quick start

In [ ]:
import openai
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [10]:
response = client.responses.create(
    model="gpt-4.1",
    input="Write a one-sentence bedtime story about a unicorn."
)

print(response.output_text)

Under a sky sprinkled with twinkling stars, a gentle unicorn named Luna danced through a magical forest, leaving trails of shimmering moonlight wherever she trotted.


### Analyze image inputs

In [13]:
response = client.responses.create(
    model="gpt-4.1",
    input=[
        {"role": "user", "content": "what teams are playing in this image?"},
        {
            "role": "user",
            "content": [
                {
                    "type": "input_image",
                    "image_url": "https://www.weeklypost.kr/news/photo/202202/3077_7598_633.jpg"
                }
            ]
        }
    ]
)

print(response.output_text)

This image shows a Sony camera, not a sports game. Therefore, there are no teams playing in this image. If you have an image of a sports event, please upload it and I'll do my best to help!


### Extend the model with tools

In [14]:
response = client.responses.create(
    model="gpt-4.1",
    tools=[{"type": "web_search_preview"}],
    input="What was a positive news story from today?"
)

print(response.output_text)

As of June 30, 2025, here are some recent positive news stories:

**Environmental Progress**

- **EU's Nature Restoration Law**: The European Union has enacted the Nature Restoration Law, aiming to restore 20% of the EU's land and sea areas by 2030 and all degraded ecosystems by 2050. This initiative seeks to combat biodiversity loss and promote ecological recovery. ([homeplanet.grove.co](https://homeplanet.grove.co/blog-posts/positive-environmental-news-stories-that-give-us-hope-in-2025?utm_source=openai))

- **Record Sea Turtle Nests in Florida**: Conservation efforts on Anna Maria Island, Florida, have led to a record-breaking 546 sea turtle nests, surpassing a 42-year-old record. Additionally, the least tern, a threatened bird species, has returned to the island for the first time in 15 years, highlighting the success of ongoing coastal preservation initiatives. ([homeplanet.grove.co](https://homeplanet.grove.co/blog-posts/positive-environmental-news-stories-that-give-us-hope-in-20

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Retrieves current weather for the given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City and country e.g. Bogotá, Colombia"
                    },
                    "units": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Units the temperature will be returned in."
                    }
                },
                "required": ["location", "units"],
                "additionalProperties": False
            }
        }
    }
]

import json

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "What's the weather in Seoul in celsius?"}
    ],
    tools=tools
)

# GPT가 함수 호출을 결정했는지 확인
if response.choices[0].message.tool_calls:
    tool_call = response.choices[0].message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    print(f"GPT wants to call function: {function_name}")
    print(f"With arguments: {arguments}")

    # 예: 실제 날씨 API 호출 등 수행 (여기선 임의 응답)
    result = {
        "location": arguments["location"],
        "temperature": "18°C",
        "condition": "Cloudy"
    }

    # 결과를 다시 GPT에게 전달
    followup = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": "What's the weather in Seoul in celsius?"},
            response.choices[0].message,  # GPT의 function call 메시지
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": json.dumps(result)
            }
        ]
    )

    print(followup.choices[0].message.content)
else:
    print(response.choices[0].message.content)

📞 GPT wants to call function: get_weather
🗺️ With arguments: {'location': 'Seoul, South Korea', 'units': 'celsius'}
The current weather in Seoul, South Korea is 18°C and cloudy.


In [ ]:
import os
import json
import requests
from openai import OpenAI

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENWEATHER_API_KEY = os.environ["OPENWEATHER_API_KEY"]  # 별도 환경변수로 관리 추천

client = OpenAI(api_key=OPENAI_API_KEY)

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Retrieves current weather for the given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City and country e.g. Bogotá, Colombia"
                    },
                    "units": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Units the temperature will be returned in."
                    }
                },
                "required": ["location", "units"],
                "additionalProperties": False
            }
        }
    }
]

def call_openweather(location: str, units: str) -> dict:
    url = "http://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": location,
        "appid": OPENWEATHER_API_KEY,
        "units": "metric" if units == "celsius" else "imperial"
    }
    response = requests.get(url, params=params)
    data = response.json()

    if response.status_code != 200 or "main" not in data:
        return {"error": f"Failed to get weather: {data.get('message', 'Unknown error')}"}

    return {
        "location": f"{data['name']}, {data['sys']['country']}",
        "temperature": f"{data['main']['temp']}°{'C' if units == 'celsius' else 'F'}",
        "condition": data['weather'][0]['description'].capitalize()
    }

# Step 1: GPT에 질문 전달
initial_response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "What's the weather in Seoul in celsius?"}
    ],
    tools=tools
)

message = initial_response.choices[0].message

if message.tool_calls:
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    weather_data = call_openweather(arguments["location"], arguments["units"])

    prompt = f"What's the weather in {arguments["location"]} in celsius?"
    
    # Step 2: 결과를 GPT에게 다시 전달
    final_response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": prompt},
            message,
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": "get_weather",
                "content": json.dumps(weather_data)
            }
        ]
    )

    print(final_response.choices[0].message.content)
else:
    print(message.content)


SyntaxError: f-string: unmatched '[' (2258349613.py, line 72)